In [ ]:
import pandas as pd
df=pd.read_csv("/content/application_train.csv")
#print(df.head())
print(df.shape)
print(df['TARGET'].value_counts(normalize=True))
print(df.isnull().sum().sort_values(ascending=False).head())
print(df.dtypes.value_counts())

(307511, 122)
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64
COMMONAREA_AVG              214865
COMMONAREA_MODE             214865
COMMONAREA_MEDI             214865
NONLIVINGAPARTMENTS_MEDI    213514
NONLIVINGAPARTMENTS_MODE    213514
dtype: int64
float64    65
int64      41
object     16
Name: count, dtype: int64


In [ ]:
# See ALL missing value percentages, not just top 5
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing_pct.head(30))

# How many columns have more than 50% missing?
print((missing_pct > 50).sum())

# Look at the categorical columns
cat_cols = df.select_dtypes(include='object').columns
print(cat_cols)
for col in cat_cols:
    print(col, ':', df[col].nunique(), 'unique values')

COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
COMMONAREA_MEDI             69.872297
NONLIVINGAPARTMENTS_MEDI    69.432963
NONLIVINGAPARTMENTS_MODE    69.432963
NONLIVINGAPARTMENTS_AVG     69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_AVG        68.354953
LIVINGAPARTMENTS_MEDI       68.354953
LIVINGAPARTMENTS_MODE       68.354953
FLOORSMIN_MODE              67.848630
FLOORSMIN_AVG               67.848630
FLOORSMIN_MEDI              67.848630
YEARS_BUILD_AVG             66.497784
YEARS_BUILD_MODE            66.497784
YEARS_BUILD_MEDI            66.497784
OWN_CAR_AGE                 65.990810
LANDAREA_MEDI               59.376738
LANDAREA_AVG                59.376738
LANDAREA_MODE               59.376738
BASEMENTAREA_MODE           58.515956
BASEMENTAREA_MEDI           58.515956
BASEMENTAREA_AVG            58.515956
EXT_SOURCE_1                56.381073
NONLIVINGAREA_MODE          55.179164
NONLIVINGAREA_AVG           55.179164
NONLIVINGARE

In [ ]:
# Step 1: Drop columns with >50% missing, EXCEPT the EXT_SOURCE columns
missing_pct = (df.isnull().sum() / len(df) * 100)
cols_to_drop = missing_pct[missing_pct > 50].index.tolist()

# Keep EXT_SOURCE columns even if they're in the drop list
ext_source_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
cols_to_drop = [col for col in cols_to_drop if col not in ext_source_cols]

df = df.drop(columns=cols_to_drop)
print("Dropped", len(cols_to_drop), "columns. New shape:", df.shape)

# Step 2: Check EXT_SOURCE missing percentages specifically
print(df[ext_source_cols].isnull().sum() / len(df) * 100)

Dropped 40 columns. New shape: (307511, 82)
EXT_SOURCE_1    56.381073
EXT_SOURCE_2     0.214626
EXT_SOURCE_3    19.825307
dtype: float64


In [ ]:
# Fill EXT_SOURCE columns with median
for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Confirm no missing values left in these columns
print(df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].isnull().sum())

# Check how many columns STILL have any missing values at all
remaining_missing = df.isnull().sum()
print(remaining_missing[remaining_missing > 0].sort_values(ascending=False))


EXT_SOURCE_1    0
EXT_SOURCE_2    0
EXT_SOURCE_3    0
dtype: int64
FLOORSMAX_AVG                   153020
FLOORSMAX_MODE                  153020
FLOORSMAX_MEDI                  153020
YEARS_BEGINEXPLUATATION_AVG     150007
YEARS_BEGINEXPLUATATION_MODE    150007
YEARS_BEGINEXPLUATATION_MEDI    150007
TOTALAREA_MODE                  148431
EMERGENCYSTATE_MODE             145755
OCCUPATION_TYPE                  96391
AMT_REQ_CREDIT_BUREAU_YEAR       41519
AMT_REQ_CREDIT_BUREAU_WEEK       41519
AMT_REQ_CREDIT_BUREAU_DAY        41519
AMT_REQ_CREDIT_BUREAU_QRT        41519
AMT_REQ_CREDIT_BUREAU_MON        41519
AMT_REQ_CREDIT_BUREAU_HOUR       41519
NAME_TYPE_SUITE                   1292
OBS_60_CNT_SOCIAL_CIRCLE          1021
DEF_30_CNT_SOCIAL_CIRCLE          1021
DEF_60_CNT_SOCIAL_CIRCLE          1021
OBS_30_CNT_SOCIAL_CIRCLE          1021
AMT_GOODS_PRICE                    278
AMT_ANNUITY                         12
CNT_FAM_MEMBERS                      2
DAYS_LAST_PHONE_CHANGE              

In [ ]:
# Group 1: drop
group1_drop = ['FLOORSMAX_AVG','FLOORSMAX_MODE','FLOORSMAX_MEDI',
               'YEARS_BEGINEXPLUATATION_AVG','YEARS_BEGINEXPLUATATION_MODE','YEARS_BEGINEXPLUATATION_MEDI',
               'TOTALAREA_MODE','EMERGENCYSTATE_MODE']
df = df.drop(columns=group1_drop)

# Group 2: fill categorical with "Unknown"
for col in ['OCCUPATION_TYPE', 'NAME_TYPE_SUITE']:
    df[col] = df[col].fillna('Unknown')

# Group 3: fill inquiry counts with 0
bureau_cols = ['AMT_REQ_CREDIT_BUREAU_YEAR','AMT_REQ_CREDIT_BUREAU_WEEK','AMT_REQ_CREDIT_BUREAU_DAY',
               'AMT_REQ_CREDIT_BUREAU_QRT','AMT_REQ_CREDIT_BUREAU_MON','AMT_REQ_CREDIT_BUREAU_HOUR']
for col in bureau_cols:
    df[col] = df[col].fillna(0)

# Group 4: fill remaining small numeric gaps with median
group4 = ['OBS_60_CNT_SOCIAL_CIRCLE','DEF_30_CNT_SOCIAL_CIRCLE','DEF_60_CNT_SOCIAL_CIRCLE',
          'OBS_30_CNT_SOCIAL_CIRCLE','AMT_GOODS_PRICE','AMT_ANNUITY','CNT_FAM_MEMBERS','DAYS_LAST_PHONE_CHANGE']
for col in group4:
    df[col] = df[col].fillna(df[col].median())

# Confirm: any missing values left anywhere?
print("Remaining missing values:", df.isnull().sum().sum())
print("Current shape:", df.shape)


Remaining missing values: 0
Current shape: (307511, 74)


In [ ]:
# Recheck which columns are still categorical after our cleanup
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(cat_cols)
for col in cat_cols:
    print(col, ':', df[col].nunique())

['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE']
NAME_CONTRACT_TYPE : 2
CODE_GENDER : 3
FLAG_OWN_CAR : 2
FLAG_OWN_REALTY : 2
NAME_TYPE_SUITE : 8
NAME_INCOME_TYPE : 8
NAME_EDUCATION_TYPE : 5
NAME_FAMILY_STATUS : 6
NAME_HOUSING_TYPE : 6
OCCUPATION_TYPE : 19
WEEKDAY_APPR_PROCESS_START : 7
ORGANIZATION_TYPE : 58


In [ ]:
# One-hot encode the low-cardinality columns
onehot_cols = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
               'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
               'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'WEEKDAY_APPR_PROCESS_START']

df = pd.get_dummies(df, columns=onehot_cols, drop_first=True)

# Frequency encode the high-cardinality columns
freq_cols = ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']
for col in freq_cols:
    freq_map = df[col].value_counts(normalize=True)   # % frequency of each category
    df[col] = df[col].map(freq_map)

# Confirm everything is now numeric
print(df.select_dtypes(include='object').columns.tolist())   # should print empty list
print(df.shape)

[]
(307511, 103)


In [ ]:
from sklearn.model_selection import train_test_split

# Separate features (X) from target (y)
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])   # SK_ID_CURR is just an ID, not a feature
y = df['TARGET']

# Stratified split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train TARGET ratio:\n", y_train.value_counts(normalize=True))
print("Test TARGET ratio:\n", y_test.value_counts(normalize=True))

Train shape: (246008, 101)
Test shape: (61503, 101)
Train TARGET ratio:
 TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64
Test TARGET ratio:
 TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Scale features (fit on train, apply same scaling to test — never fit on test data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train baseline model
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train_scaled, y_train)

print("Model trained.")

Model trained.


In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Get predictions
y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]   # probability of class 1 (default)

# ROC-AUC
auc = roc_auc_score(y_test, y_pred_proba)
print("ROC-AUC:", auc)

# Precision, recall, F1 for both classes
print(classification_report(y_test, y_pred))

# Confusion matrix
print(confusion_matrix(y_test, y_pred))

ROC-AUC: 0.7475100474270404
              precision    recall  f1-score   support

           0       0.96      0.69      0.80     56538
           1       0.16      0.67      0.26      4965

    accuracy                           0.69     61503
   macro avg       0.56      0.68      0.53     61503
weighted avg       0.90      0.69      0.76     61503

[[38997 17541]
 [ 1615  3350]]
